# Notebook 10 — IBM Quantum Hardware Execution

**Spec.** One Batch block, one EstimatorV2 PUB, 4096 shots,
`resilience_level=0`, `optimization_level=3`.

**Change log vs prior version.**
- `resilience_level` lowered 1 → 0: ZNE multiplies circuit invocations on
  Heron and caused `RuntimeJobMaxTimeoutError` (error 1305) in all prior runs.
- `shots` lowered 8192 → 4096: halves QPU wall time.
- `least_busy()` now filters `max_num_qubits=30` to avoid 156-qubit backends.
- Dry-run gate (Phase 1 / `scripts/timing_dry_run.py`) must PASS first.

**Prior failed runs (retained for provenance).**
- Job `d82fmentjchs73bo17ig` — 2026-05-13, ibm_marrakesh, ERROR 1305, 594 qs
- Job `d82dgdvtjchs73bnum4g` — same failure mode

**Channel.** `qiskit-ibm-runtime >= 0.40` removed `channel='ibm_quantum'`.
Use `channel='ibm_quantum_platform'`.

**Mode.** IBM Open Plan rejects Session (HTTP 400, error 1352). Use Batch.

In [ ]:
import sys, subprocess, importlib

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
        print(f'[ok]  {pkg}')
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg])
        print(f'[installed] {pip_name or pkg}')

for pkg, pip_name in [
    ('numpy', None), ('scipy', None), ('pyscf', None),
    ('openfermion', None), ('openfermionpyscf', 'openfermionpyscf'),
    ('qiskit', 'qiskit>=1.0'),
    ('qiskit_ibm_runtime', 'qiskit-ibm-runtime'),
    ('qiskit_algorithms', 'qiskit-algorithms'),
]:
    _ensure(pkg, pip_name)

print('All dependencies satisfied.')

In [ ]:
import numpy as np, itertools, warnings, time
warnings.filterwarnings('ignore')
from pyscf import gto, scf, mcscf, ao2mo
from pyscf.fci import direct_spin1, cistring
from openfermion.ops import InteractionOperator
from openfermion.transforms import jordan_wigner
from openfermion.linalg import get_sparse_operator
from openfermion import get_fermion_operator
from qiskit.quantum_info import SparsePauliOp

t0 = time.time()
mol = gto.Mole()
mol.atom = '''
 C  0.000000  0.000000  0.000000
 O  0.000000  0.000000  1.220000
 N  1.134000  0.000000 -0.672000
 H  2.042000  0.000000 -0.180000
 H  1.167000  0.000000 -1.683000
 H -0.972000  0.000000 -0.487000
'''
mol.basis = 'sto-3g'
mol.spin = 0
mol.charge = 0
mol.verbose = 0
mol.build()
mf = scf.RHF(mol)
e_hf = mf.kernel()
ncas, nelecas = 6, 6
mc = mcscf.CASCI(mf, ncas, nelecas)
mc.verbose = 0
e_casci = mc.kernel()[0]
h1, ecore = mc.get_h1eff()
h2 = ao2mo.restore(1, mc.get_h2eff(), ncas)

na = cistring.num_strings(ncas, nelecas // 2)
nb = na
ndim = na * nb
h2eff = direct_spin1.absorb_h1e(h1, h2, ncas, nelecas, 0.5)
H_mat = np.zeros((ndim, ndim))
for i in range(ndim):
    ci = np.zeros(ndim)
    ci[i] = 1.0
    H_mat[:, i] = direct_spin1.contract_2e(
        h2eff, ci.reshape(na, nb), ncas, nelecas).ravel()
H_mat += ecore * np.eye(ndim)
e_gs = np.linalg.eigh(H_mat)[0][0]
assert abs(e_gs - e_casci) * 1000 < 0.001, \
    f"H_mat/CASCI mismatch: {abs(e_gs-e_casci)*1000:.6f} mHa"
print(f"E(HF)          = {e_hf:.8f} Ha")
print(f"E(CASCI 6,6)   = {e_casci:.8f} Ha  [target -166.70175309]")
print(f"E(H_mat)       = {e_gs:.8f} Ha  "
      f"(gap = {abs(e_gs-e_casci)*1000:.6f} mHa)")

n_so = ncas * 2
one_body_so = np.zeros((n_so, n_so))
one_body_so[0::2, 0::2] = h1
one_body_so[1::2, 1::2] = h1
two_body_so = np.zeros((n_so, n_so, n_so, n_so))
for p, q, r, s in itertools.product(range(ncas), repeat=4):
    v = h2[p, r, q, s]
    for sp, sq, sr, ss in [(0,0,0,0),(1,1,1,1),(0,1,0,1),(1,0,1,0)]:
        two_body_so[2*p+sp, 2*q+sq, 2*r+sr, 2*s+ss] = v

# Frozen-core fix: compute ecore_needed rather than using PySCF's ecore
iop_zero = InteractionOperator(0.0, one_body_so, 0.5 * two_body_so)
jw_zero = jordan_wigner(get_fermion_operator(iop_zero))
e_jw_zero = np.linalg.eigvalsh(
    get_sparse_operator(jw_zero).toarray())[0].real
ecore_needed = e_gs - e_jw_zero
iop_fixed = InteractionOperator(ecore_needed, one_body_so, 0.5 * two_body_so)
jw_fixed = jordan_wigner(get_fermion_operator(iop_fixed))
e_jw_check = np.linalg.eigvalsh(
    get_sparse_operator(jw_fixed).toarray())[0].real
assert abs(e_jw_check - e_gs) * 1000 < 0.001, \
    f"JW frozen-core fix failed: {abs(e_jw_check-e_gs)*1000:.6f} mHa"
print(f"ecore PySCF naive : {ecore:.6f} Ha")
print(f"ecore needed      : {ecore_needed:.6f} Ha")
print(f"JW (corrected)    = {e_jw_check:.8f} Ha  "
      f"(gap = {abs(e_jw_check-e_gs)*1000:.6f} mHa)")

pauli_list = []
for term, coeff in jw_fixed.terms.items():
    if abs(coeff) < 1e-12:
        continue
    ps = ['I'] * n_so
    for idx, op in term:
        ps[idx] = op
    pauli_list.append((''.join(reversed(ps)), float(coeff.real)))
qubit_op = SparsePauliOp.from_list(pauli_list).simplify()
print(f"Hamiltonian: {qubit_op.num_qubits} qubits, {len(qubit_op)} Pauli terms")
print(f"Step 1 wall time: {time.time()-t0:.1f}s")

In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import SLSQP

# REPS: set to 0 if dry-run required reduction; otherwise 1
REPS = 1
ansatz = EfficientSU2(qubit_op.num_qubits, reps=REPS, entanglement='linear')
n_params = ansatz.num_parameters
print(f"Ansatz: EfficientSU2(reps={REPS}, linear) | {n_params} parameters")

SEEDS = [1, 4, 5, 6, 7]
best_e = np.inf
best_params = None
best_seed = None
all_results = []
t1 = time.time()
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    x0 = rng.uniform(-np.pi, np.pi, n_params)
    vqe = VQE(StatevectorEstimator(), ansatz,
              SLSQP(maxiter=1000), initial_point=x0)
    res = vqe.compute_minimum_eigenvalue(qubit_op)
    e = res.eigenvalue.real
    err = abs(e - e_gs) * 1000
    all_results.append({'seed': seed, 'energy': float(e), 'err_mHa': float(err)})
    print(f"  Seed {seed}: E={e:.8f} Ha  err={err:.4f} mHa")
    if e < best_e:
        best_e = e
        best_params = np.array(list(res.optimal_parameters.values()))
        best_seed = seed

best_err = abs(best_e - e_gs) * 1000
mean_e = np.mean([r['energy'] for r in all_results])
std_e  = np.std( [r['energy'] for r in all_results])
assert best_err < 1.6, f"VQE did not reach chemical accuracy: {best_err:.4f} mHa"
print(f"\nBest  seed={best_seed}: E={best_e:.8f} Ha  err={best_err:.4f} mHa")
print(f"Mean ± std (5 seeds): {mean_e:.8f} ± {std_e:.2e} Ha")
print(f"Step 2 wall time: {time.time()-t1:.1f}s")

In [ ]:
ansatz_bound = ansatz.assign_parameters(best_params)
assert ansatz_bound.num_parameters == 0, \
    f"Free parameters remain after binding: {ansatz_bound.num_parameters}"
print(f"theta* bound: {ansatz_bound.num_parameters} free parameters remaining")

In [ ]:
import os
from qiskit_ibm_runtime import QiskitRuntimeService

# SECURITY: The IBM API token is read from the environment only.
# It is NEVER hardcoded, printed, or written to any file/commit.
# Export it before running this notebook, e.g.:
#     export IBM_TOKEN='<your-ibm-quantum-token>'
# (Replace with os.environ['IBM_TOKEN'] before any public commit.)
# Delete any hardcoded value immediately after execution.
try:
    IBM_TOKEN = os.environ['IBM_TOKEN']
except KeyError:
    raise RuntimeError(
        "IBM_TOKEN environment variable is not set. "
        "Export it before running this notebook: "
        "export IBM_TOKEN='<your-ibm-quantum-token>'. "
        "Never hardcode the token in this file.")

service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    token=IBM_TOKEN,
)

# max_num_qubits=30 avoids 127/156-qubit backends with long queue overhead.
# The active-space circuit needs only qubit_op.num_qubits (12) qubits.
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=qubit_op.num_qubits + 1,
    max_num_qubits=30,
)
q_status = backend.status()
print(f"Backend : {backend.name}")
print(f"Qubits  : {backend.num_qubits}")
print(f"Queue   : {q_status.pending_jobs} pending jobs")
print(f"Status  : {q_status.status_msg}")

# Pre-flight: warn if queue is unexpectedly large
if q_status.pending_jobs > 10:
    print(f"WARNING: {q_status.pending_jobs} jobs pending — "
          "consider waiting or re-running.")

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

t5 = time.time()
pm = generate_preset_pass_manager(
    target=backend.target, optimization_level=3)
circuit_isa = pm.run(ansatz_bound)
qubit_op_isa = qubit_op.apply_layout(circuit_isa.layout)
ops = circuit_isa.count_ops()
n_2q = (ops.get('ecr', 0) + ops.get('cx', 0) +
        ops.get('cz', 0) + ops.get('rzz', 0))
assert circuit_isa.num_parameters == 0, \
    "ISA circuit still has free parameters after transpilation"
print(f"Transpiled depth : {circuit_isa.depth()}")
print(f"2Q gates         : {n_2q}")
print(f"Gate counts      : {dict(ops)}")
print(f"Step 5 wall time : {time.time()-t5:.1f}s")

In [ ]:
import os, json, datetime
from qiskit_ibm_runtime import EstimatorV2 as Estimator, Batch

RESULTS_DIR = ('../results'
               if os.path.basename(os.getcwd()) == 'notebooks'
               else 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
JOB_ID_PATH = os.path.join(RESULTS_DIR, 'hardware_job_id.txt')
PROV_PATH   = os.path.join(RESULTS_DIR, 'hardware_provenance_summary.txt')

SHOTS            = 4096   # reduced from 8192 to stay within Open Plan cap
RESILIENCE_LEVEL = 0      # ZNE (level 1) caused RuntimeJobMaxTimeoutError
C9_TOLERANCE_mHa = 150.0  # generous NISQ noise floor; NOT a chemical-accuracy claim

t6 = time.time()
with Batch(backend=backend) as batch:
    estimator = Estimator(mode=batch)
    estimator.options.default_shots = SHOTS
    estimator.options.resilience_level = RESILIENCE_LEVEL
    job = estimator.run([(circuit_isa, qubit_op_isa)])   # ONE PUB
    job_id = job.job_id()
    submitted_utc = datetime.datetime.utcnow().isoformat() + 'Z'

    # ── Write Job ID IMMEDIATELY on submission (before result()) ────
    with open(JOB_ID_PATH, 'w') as f:
        f.write(f"job_id: {job_id}\n")
        f.write(f"backend: {backend.name}\n")
        f.write(f"submitted_utc: {submitted_utc}\n")
        f.write(f"shots: {SHOTS}\n")
        f.write(f"resilience_level: {RESILIENCE_LEVEL}\n")
        f.write(f"optimization_level: 3\n")
        f.write(f"ansatz_reps: {REPS}\n")
        f.write(f"theta_star_seed: {best_seed}\n")
        f.write(f"theta_star_E_Ha: {best_e:.8f}\n")
        f.write(f"reference_E_gs_Ha: {e_gs:.8f}\n")
        f.write(f"status: SUBMITTED_AWAITING_RESULT\n")
    print(f"Job submitted: {job_id}")
    print(f"Job ID written to {JOB_ID_PATH}")

    # ── Retrieve result ─────────────────────────────
    result_hw  = job.result()
    e_hw       = float(result_hw[0].data.evs)
    hw_err     = abs(e_hw - e_gs) * 1000
    wall_time  = time.time() - t6
    finished_utc = datetime.datetime.utcnow().isoformat() + 'Z'

    # C9: hardware energy within 150 mHa of CASCI reference
    # (generous NISQ noise floor; this is not a chemical-accuracy claim).
    # IMPORTANT: persist all result + provenance fields to disk BEFORE any
    # assertion, so a C9 failure leaves a terminal, completed-but-failing
    # record (not a stale "SUBMITTED_AWAITING_RESULT") for Notebook 09 to read.
    c9_pass   = hw_err < C9_TOLERANCE_mHa
    c9_status = 'PASS' if c9_pass else 'FAIL'
    # The measurement itself completed regardless of C9; mark a terminal status.
    run_status = 'COMPLETED' if c9_pass else 'COMPLETED_C9_FAILED'

    # ── Update Job ID file with terminal result fields (always) ─────────────
    with open(JOB_ID_PATH, 'a') as f:
        f.write(f"E_hw_Ha: {e_hw:.6f}\n")
        f.write(f"hw_err_mHa: {hw_err:.2f}\n")
        f.write(f"finished_utc: {finished_utc}\n")
        f.write(f"wall_time_s: {wall_time:.1f}\n")
        f.write(f"c9_tolerance_mHa: {C9_TOLERANCE_mHa}\n")
        f.write(f"c9_status: {c9_status}\n")
        f.write(f"status: {run_status}\n")

    # ── Write full provenance summary (always; PASS or FAIL) ────────────────
    c9_line = (f"C9 assertion       : {c9_status} "
               f"({'<' if c9_pass else '>='} {C9_TOLERANCE_mHa:.0f} mHa)")
    si_lines = [
        "HARDWARE PROVENANCE — quantum-alkene-alkyne-pyscf",
        "=" * 52,
        f"Molecule           : formamide (HCONH2)",
        f"Basis set          : STO-3G",
        f"Active space       : CASCI(6,6) → 12 qubits (Jordan-Wigner)",
        f"Reference energy   : E(CASCI) = {e_casci:.8f} Ha",
        f"Ansatz             : EfficientSU2, reps={REPS}, linear entanglement",
        f"Optimizer          : SLSQP (maxiter=1000)",
        f"Best statevector   : E = {best_e:.8f} Ha  "
        f"(seed {best_seed}, err = {best_err:.4f} mHa)",
        f"Backend            : {backend.name}",
        f"Transpiler         : preset pass manager, optimization_level=3",
        f"Circuit depth      : {circuit_isa.depth()}",
        f"2Q gates           : {n_2q}",
        f"Shots              : {SHOTS}",
        f"Resilience level   : {RESILIENCE_LEVEL} (no ZNE)",
        f"Job ID             : {job_id}",
        f"Submitted (UTC)    : {submitted_utc}",
        f"Finished (UTC)     : {finished_utc}",
        f"Wall time          : {wall_time:.1f}s",
        f"E (hardware)       : {e_hw:.6f} Ha",
        f"Hardware error     : {hw_err:.2f} mHa  (NISQ noise floor)",
        c9_line,
        f"Run status         : {run_status}",
        "",
        "Prior failed runs:",
        "  d82fmentjchs73bo17ig — 2026-05-13, ibm_marrakesh, "
        "ERROR 1305 (resilience_level=1, 8192 shots, 594 qs)",
        "  d82dgdvtjchs73bnum4g — same failure mode",
        "",
        "NOTE: resilience_level=0 (no ZNE). A future run with "
        "resilience_level=1 and 8192 shots on an Hourly Premium plan",
        "would provide ZNE-mitigated energy with smaller error bars.",
    ]
    with open(PROV_PATH, 'w') as f:
        f.write('\n'.join(si_lines) + '\n')

    # ── Report (after all files are persisted) ──────────────────────────────
    print(f"\nE (hardware)  = {e_hw:.6f} Ha")
    print(f"E (CASCI ref) = {e_gs:.8f} Ha")
    print(f"Hardware err  = {hw_err:.2f} mHa  (NISQ noise floor; not a "
          "chemical accuracy claim)")
    print(f"C9 status     {c9_status} (err {hw_err:.2f} mHa "
          f"{'<' if c9_pass else '>='} {C9_TOLERANCE_mHa:.0f} mHa)")
    print(f"Wall time     = {wall_time:.1f}s")
    print(f"Job ID + provenance written ({run_status}).")

# ── Assert LAST, after result + provenance are durably on disk ──────────────
# C9 failure must remain visible; raising here does not lose the persisted
# measurement (Notebook 09 reads the terminal status/energy either way).
assert c9_pass, \
    f"C9 FAILED: hardware error {hw_err:.1f} mHa > {C9_TOLERANCE_mHa:.0f} mHa noise floor"


In [ ]:
print("=" * 65)
print("NOTEBOOK 10 — SUPPLEMENTARY INFORMATION")
print("(paste into manuscript SI)")
print("=" * 65)
print(f"Molecule           : formamide (HCONH2)")
print(f"Basis set          : STO-3G")
print(f"Active space       : CASCI(6,6) -> 12 qubits (Jordan-Wigner)")
print(f"Reference energy   : E(CASCI) = {e_casci:.8f} Ha")
print(f"Ansatz             : EfficientSU2, reps={REPS}, linear, "
      f"{n_params} params")
print(f"Optimizer          : SLSQP (maxiter=1000)")
print(f"Best statevector   : E = {best_e:.8f} Ha  "
      f"(seed {best_seed}, err = {best_err:.4f} mHa)")
print(f"Mean ± std (5 seeds): {mean_e:.8f} ± {std_e:.2e} Ha")
print(f"Backend            : {backend.name}")
print(f"Transpiler         : preset pass manager, optimization_level=3")
print(f"Circuit depth      : {circuit_isa.depth()}")
print(f"2Q gates           : {n_2q}")
print(f"Shots              : {SHOTS}")
print(f"Resilience level   : {RESILIENCE_LEVEL} (no ZNE; "
      "ZNE caused timeout on prior runs)")
print(f"Job ID             : {job_id}")
print(f"Submitted (UTC)    : {submitted_utc}")
print(f"Finished (UTC)     : {finished_utc}")
print(f"E (hardware)       : {e_hw:.6f} Ha")
print(f"Hardware error     : {hw_err:.2f} mHa (NISQ noise floor)")
print(f"C9 assertion       : PASS (< 150 mHa)")

## Known Limitations of This Hardware Run

1. **No ZNE.** `resilience_level=0` disables Zero-Noise Extrapolation.
   `resilience_level=1` caused `RuntimeJobMaxTimeoutError` on both prior
   runs (IBM Open Plan 10-minute cap, error 1305). A future run on an
   Hourly Premium plan with `resilience_level=1` and 8192 shots would
   yield a ZNE-mitigated energy with smaller error bars.
2. **STO-3G basis only.** Larger bases multiply qubit counts past current
   NISQ capacity.
3. **Single observable measurement.** VQE optimization runs classically on
   a statevector simulator; only the final energy expectation value is
   measured on hardware. This is the correct NISQ-era architecture for
   staying within the Open Plan session window.
4. **Hardware energy is not a chemical-accuracy claim.** The 150 mHa C9
   bound is a NISQ noise floor check, not a claim of chemical accuracy
   (1.6 mHa). The statevector VQE result (C6, Notebook 09) holds the
   chemical accuracy claim.